# SETUP

In [1]:
# import the functions from collector.py
import sys
import os
sys.path.append(os.path.abspath(".."))

from bjarne_api.collector_new import Fetcher
import pandas as pd
from functions import filter_train_type, get_possible_destinations, get_connections, create_features

# API CALL

In [5]:
### DEFINE JOURNEY
input = "Göttingen" 

### GET INFOS ON ALL RIDES
# Ausführung im Hauptprogramm 
if __name__ == "__main__":
    fetcher = Fetcher()
    station_name = input
    
    print(f"Suche Station ID für {station_name}...")
    station_id = fetcher.get_station_id(station_name)
    
    if station_id:
        print(f"Station ID gefunden: {station_id}. Suche Fernverkehrsabfahrten (nächste 60min)...")
        departures = fetcher.stations_details(station_id)
        
        if departures:
            print(f"{len(departures)} Abfahrten gefunden. Lade Details...")
            
            all_trips_dfs = []
            
            
            for i, departure in enumerate(departures, start=1):
                if "tripId" in departure:
                    trip_id = departure["tripId"]
                    line_name = departure.get("line", {}).get("name", "Unbekannt")
                    
                    print(f"Lade Trip {i}/{len(departures)}: {line_name}")
                    
                    trip_data = fetcher.trip_details(trip_id)
                    if trip_data is not None:
                        df_trip = fetcher.create_dataframe(trip_data, ride_id=i)
                    
                    if not df_trip.empty:
                        all_trips_dfs.append(df_trip)
                        
                    else: 
                        print(f"Details für Trip {line_name} konnten nicht verarbeitete werden")
            
            # Zusammenfügen aller DataFrames
            if all_trips_dfs:
                final_df = pd.concat(all_trips_dfs, ignore_index=True)
                print("\n--- FERTIG ---")
                print(final_df)
                print(f"\nGesamtanzahl Haltepunkte analysiert: {len(final_df)}")
                print(f"Anzahl Züge: {final_df['ride_id'].nunique()}")
            else:
                print("Konnte keine Trip-Details verarbeiten.")
        else:
            print("Keine passenden Abfahrten im Zeitraum gefunden.")
    else:
        print("Station nicht gefunden.")




Suche Station ID für Göttingen...
Fehler beim Abrufen der Stations-ID: 503 Server Error: Service Unavailable for url: https://v6.db.transport.rest/locations?query=G%C3%B6ttingen&type=station&results=1
Station nicht gefunden.


# FILTER AND SELECT DATA

In [5]:
# filter data
df_filtered = filter_train_type(final_df)

# get valid destinations (list format)
ls_destinations = get_possible_destinations(df_filtered, input)

# select destination
destination = "Berlin Hbf" # COMES FROM INPUT IN STREAMLIT (USER SHOULD CHOOSE)

# get rides between stations
df_trip = get_connections(df_filtered, input, destination)
df_trip = df_trip.drop(columns = ["current_delay", "stops_total", "stop_index", "stops_remaining"])

# CREATE FEATURES

In [6]:
df_features = create_features(df_trip)